# Backtesting

Imports

In [28]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import statsmodels.api as sm

### Step 1: Load cointegration and price data

In [29]:
data_folder_path = Path("data")
print(data_folder_path.exists())

True


In [30]:
bh_corrected_df = pd.read_parquet(data_folder_path / "bh_corrected_results.parquet")
bh_sig_results_df = bh_corrected_df[bh_corrected_df["bh_reject"]]
sig_results_df = pd.read_parquet(data_folder_path / "sig_results.parquet")

In [31]:
print(sig_results_df.head())

                                 correlation   coint_t   p_value  \
formation_end ticker_0 ticker_1                                    
2016-01-01    STT      GS           0.753596 -3.362022  0.046747   
                       MS           0.739525 -4.778833  0.000411   
                       LNC          0.776851 -3.481309  0.034089   
              COP      EOG          0.781141 -3.454777  0.036620   
              KIM      O            0.762781 -4.339056  0.002224   

                                 critical_1pct  critical_5pct  critical_10pct  \
formation_end ticker_0 ticker_1                                                 
2016-01-01    STT      GS            -3.918346      -3.348304       -3.052893   
                       MS            -3.918346      -3.348304       -3.052893   
                       LNC           -3.918346      -3.348304       -3.052893   
              COP      EOG           -3.918346      -3.348304       -3.052893   
              KIM      O             

In [32]:
print(bh_corrected_df.head())

                                 correlation   coint_t   p_value  \
formation_end ticker_0 ticker_1                                    
2016-01-01    SWKS     AVGO         0.705393 -0.019827  0.985429   
              BAC      STT          0.730833 -2.431513  0.310079   
                       HBAN         0.767496 -2.654967  0.216192   
                       PFG          0.710855 -2.270825  0.388197   
                       TFC          0.756456 -2.563165  0.251944   

                                 critical_1pct  critical_5pct  critical_10pct  \
formation_end ticker_0 ticker_1                                                 
2016-01-01    SWKS     AVGO          -3.918346      -3.348304       -3.052893   
              BAC      STT           -3.918346      -3.348304       -3.052893   
                       HBAN          -3.918346      -3.348304       -3.052893   
                       PFG           -3.918346      -3.348304       -3.052893   
                       TFC           

In [33]:
historical_ohlcv_df = pd.read_parquet(data_folder_path / "historical_ohlcv_df.parquet")

In [34]:
print(historical_ohlcv_df.head())

Ticker     SOLV                       KVUE                        ...  \
Price      Open High Low Close Volume Open High Low Close Volume  ...   
Date                                                              ...   
2014-01-02  NaN  NaN NaN   NaN    NaN  NaN  NaN NaN   NaN    NaN  ...   
2014-01-03  NaN  NaN NaN   NaN    NaN  NaN  NaN NaN   NaN    NaN  ...   
2014-01-06  NaN  NaN NaN   NaN    NaN  NaN  NaN NaN   NaN    NaN  ...   
2014-01-07  NaN  NaN NaN   NaN    NaN  NaN  NaN NaN   NaN    NaN  ...   
2014-01-08  NaN  NaN NaN   NaN    NaN  NaN  NaN NaN   NaN    NaN  ...   

Ticker              R                                                 EOG  \
Price            Open       High        Low      Close  Volume       Open   
Date                                                                        
2014-01-02  51.244708  51.384891  50.683965  50.915272  453500  59.113503   
2014-01-03  51.055471  51.335842  50.838185  51.020428  303500  58.822462   
2014-01-06  51.265752  51.4479

### Step 2: Estimate relationship parameters for all pairs

In [35]:
close_prices = historical_ohlcv_df.xs("Close", level="Price", axis=1)
print(close_prices.head())

Ticker      SOLV  KVUE  SNI  LLL  WBA  EXE       CBRE         MTD  ADT  \
Date                                                                     
2014-01-02   NaN   NaN  NaN  NaN  NaN  NaN  26.340000  241.529999  NaN   
2014-01-03   NaN   NaN  NaN  NaN  NaN  NaN  26.400000  242.710007  NaN   
2014-01-06   NaN   NaN  NaN  NaN  NaN  NaN  26.250000  242.419998  NaN   
2014-01-07   NaN   NaN  NaN  NaN  NaN  NaN  26.190001  248.360001  NaN   
2014-01-08   NaN   NaN  NaN  NaN  NaN  NaN  26.190001  249.669998  NaN   

Ticker            IQV  ...        GIS   AMD        CAG       MKTX       SCHW  \
Date                   ...                                                     
2014-01-02  45.799999  ...  31.121460  3.95  16.424904  57.739132  22.172007   
2014-01-03  46.290001  ...  31.033251  4.00  16.385775  59.292953  22.240696   
2014-01-06  46.029999  ...  31.083658  4.13  16.307476  57.483101  22.163406   
2014-01-07  46.790001  ...  31.404943  4.18  16.586372  58.030479  21.931561   
2

Create a function to estimate params for a pair

In [36]:
def estimate_params(ticker_0, ticker_1, formation_start, formation_end):
    pair_prices = close_prices.loc[
        formation_start:(formation_end - pd.DateOffset(days=1)),
        [ticker_0, ticker_1]
    ].dropna()

    y = pair_prices[ticker_0]
    x = pair_prices[ticker_1]

    X = sm.add_constant(x)
    model = sm.OLS(y, X).fit()

    alpha = model.params["const"]
    beta = model.params[ticker_1]

    spread = y - (alpha + beta * x)

    return alpha, beta, spread.mean(), spread.std()

In [37]:
print(estimate_params("AAPL", "MSFT", pd.Timestamp("2020-01-01"), pd.Timestamp("2020-12-31")))

(np.float64(-70.92463045076721), np.float64(0.8908157857894756), np.float64(1.0331065808829393e-13), np.float64(7.600241702353187))


Create a function to estimate params for all pairs in a window and store in a dataframe

In [54]:
def estimate_params_for_window(pairs, formation_start, formation_end):
    results = []

    for ticker_0, ticker_1 in pairs:
        alpha, beta, spread_mean, spread_std = estimate_params(ticker_0, ticker_1, formation_start, formation_end)
        results.append({
            "formation_end": formation_end,
            "ticker_0": ticker_0,
            "ticker_1": ticker_1,
            "alpha": alpha,
            "beta": beta,
            "spread_mean": spread_mean,
            "spread_std": spread_std
        })
    return pd.DataFrame(results).set_index(["formation_end", "ticker_0", "ticker_1"])

Create a function to estimate params for all windows in a dataframe of cointegration data. We will call this function twice: once on the raw p-value threshold data and once on the BH-corrected data.

In [57]:
def estimate_params_for_all_windows(coint_df, window_size=504):
    results = []

    formation_ends = coint_df.index.get_level_values("formation_end").unique()

    for formation_end in formation_ends:
        end_idx = close_prices.index.searchsorted(formation_end, side="left") - 1
        formation_start = close_prices.index[end_idx - (window_size - 1)]

        pairs = coint_df.loc[formation_end].index.tolist()

        window_params_df = estimate_params_for_window(
            pairs,
            formation_start,
            formation_end
        )

        results.append(window_params_df)

    return pd.concat(results, ignore_index=False)

Estimate params for both sig_results_df and bh_corrected_df

In [ ]:
sig_window_params_df = estimate_params_for_all_windows(sig_results_df)
bh_window_params_df = estimate_params_for_all_windows(bh_sig_results_df)

In [59]:
sig_window_params_df.head()

alpha      beta   spread_mean  spread_std
formation_end ticker_0 ticker_1                                               
2016-01-01    STT      GS        15.529639  0.254871  4.398598e-15    1.591031
                       MS        15.050690  1.496055  4.638265e-15    1.330992
                       LNC        9.530159  1.223589  4.391549e-14    1.630143
              COP      EOG       -8.118959  0.813194  1.339317e-13    2.007876
              KIM      O          0.943563  0.495747 -4.854670e-14    0.274850

In [60]:
bh_window_params_df

alpha      beta   spread_mean  spread_std
formation_end ticker_0 ticker_1                                               
2016-01-01    XEL      CMS        3.514142  0.847090  5.465822e-14    0.228813
2016-02-01    XEL      CMS        3.464461  0.849943  4.194881e-14    0.236661
2016-03-01    STT      MS        10.544876  1.664574 -8.238912e-14    1.125561
              XEL      CMS        3.481574  0.849849 -7.390913e-14    0.234388
2016-04-01    STT      MS        11.365390  1.633517 -7.793413e-14    1.122996
...                                    ...       ...           ...         ...
2025-11-01    PPL      NI         8.675948  0.647139  3.207311e-15    0.611207
              CPT      ESS        3.367900  0.392093 -2.774218e-13    2.017918
              RF       KEY       -1.634339  1.484203  2.443196e-14    0.445216
2025-12-01    CPT      ESS        2.656230  0.394576 -7.993606e-14    2.043802
              RF       KEY       -2.206338  1.521070  1.646655e-14    0.419373

[406 rows x 4 columns]

### Simulate trading using formation window data

Calculate z-score from 2 prices

In [43]:
def calculate_z_score(price_0, price_1, alpha, beta, spread_mean, spread_std):
    spread = alpha + beta * price_1 - price_0 # y is ticker_0, x is ticker_1 according to our earlier code
    return (spread - spread_mean) / spread_std

In [ ]:
trade_thresh = 2 # only trade if absolute value of z-score is > 2
sell_thresh = 0.5 # we will sell our current portfolio if absolute value of z score < 0.5

Function to trade within a window for a given df

In [94]:
def simulate_trading_window(params_df, formation_end):
    trading_ohlcv_df = historical_ohlcv_df.loc[
        (historical_ohlcv_df.index >= formation_end) &
        (historical_ohlcv_df.index <= formation_end + pd.DateOffset(months=1))
    ]

    trading_open_df = trading_ohlcv_df.xs('Open', level='Price', axis=1).iloc[:-1] # used for signals
    trading_close_df = trading_ohlcv_df.xs('Close', level='Price', axis=1).iloc[1:] # used for returns

    print(trading_close_df)

In [95]:
simulate_trading_window(sig_window_params_df, pd.Timestamp("2016-01-01"))

Ticker      SOLV  KVUE  SNI  LLL  WBA  EXE       CBRE         MTD  ADT  \
Date                                                                     
2016-01-05   NaN   NaN  NaN  NaN  NaN  NaN  34.389999  331.380005  NaN   
2016-01-06   NaN   NaN  NaN  NaN  NaN  NaN  32.910000  324.739990  NaN   
2016-01-07   NaN   NaN  NaN  NaN  NaN  NaN  31.070000  315.730011  NaN   
2016-01-08   NaN   NaN  NaN  NaN  NaN  NaN  30.719999  313.929993  NaN   
2016-01-11   NaN   NaN  NaN  NaN  NaN  NaN  30.090000  310.859985  NaN   
2016-01-12   NaN   NaN  NaN  NaN  NaN  NaN  30.330000  317.019989  NaN   
2016-01-13   NaN   NaN  NaN  NaN  NaN  NaN  29.090000  311.579987  NaN   
2016-01-14   NaN   NaN  NaN  NaN  NaN  NaN  29.469999  310.660004  NaN   
2016-01-15   NaN   NaN  NaN  NaN  NaN  NaN  28.940001  313.619995  NaN   
2016-01-19   NaN   NaN  NaN  NaN  NaN  NaN  28.559999  308.429993  NaN   
2016-01-20   NaN   NaN  NaN  NaN  NaN  NaN  28.010000  307.450012  NaN   
2016-01-21   NaN   NaN  NaN  NaN  NaN 